## 부품마스터 EDA

담당: 황유진 관련이슈: #47

| 컬럼명              | 의미       |
| ---------------- | -------- |
| `PART_CD`        | 부품 코드    |
| `PART_NM`        | 부품명      |
| `PART_TYPE`      | 부품 유형    |
| `APPLY_EQP_TYPE` | 적용 설비 유형 |
| `STD_LIFE_H`     | 표준 수명    |
| `LEAD_TIME_D`    | 리드타임     |


- 시간 범위: 부품마스터에는 시간 컬럼이 없어 해당 없음
- 시간 조인: 부품마스터 자체는 기존 TC/OP 데이터와 시간 기준으로 직접 조인할 수 없음
- 키 조인: `PART_CD` 를 이용해 `G-02_부품교체이력.csv` 와 연결 가능 여부를 확인

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/G-01_부품마스터.csv",
    encoding="utf-8-sig"
)

print("행 수:", len(df))
print("컬럼 수:", len(df.columns))
print("컬럼명:")
print(df.columns.tolist())

In [ ]:
print("자료형:")
print(df.dtypes)

print("\n결측값:")
print(df.isna().sum())

print("\n중복 행 수:")
print(df.duplicated().sum())

print("\nPART_CD 고유값 수:", df["PART_CD"].nunique())
print("전체 행 수:", len(df))
print("PART_CD 중복 수:", df["PART_CD"].duplicated().sum())

- 전체 129개 부품에 결측값과 중복 행이 존재하지 않는다.
- `PART_CD`의 고유값 수는 전체 행 수와 동일한 129개이며 중복된 부품 코드는 없다.
- 따라서 `PART_CD`는 부품을 구분하는 식별자로 사용할 수 있다.
- `STD_LIFE_H`, `LEAD_TIME_D`는 수치형으로 저장되어 있어 범위 및 분포 분석이 가능하다.

In [ ]:
print("PART_TYPE 분포:")
print(df["PART_TYPE"].value_counts())

In [ ]:
eqp_type_count = (
    df["APPLY_EQP_TYPE"]
    .value_counts()
    .rename_axis("APPLY_EQP_TYPE")
    .reset_index(name="count")
)

display(eqp_type_count)

In [ ]:
print("STD_LIFE_H 요약:")
print(df["STD_LIFE_H"].describe())

print("\nLEAD_TIME_D 요약:")
print(df["LEAD_TIME_D"].describe())

print("\nSTD_LIFE_H 중앙값:", df["STD_LIFE_H"].median())
print("LEAD_TIME_D 중앙값:", df["LEAD_TIME_D"].median())

In [ ]:
import matplotlib.pyplot as plt

df["STD_LIFE_H"].hist(bins=20)
plt.title("STD_LIFE_H Distribution")
plt.xlabel("STD_LIFE_H")
plt.ylabel("Count")
plt.show()

df["LEAD_TIME_D"].hist(bins=20)
plt.title("LEAD_TIME_D Distribution")
plt.xlabel("LEAD_TIME_D")
plt.ylabel("Count")
plt.show()

In [ ]:
chg = pd.read_csv(
    "../data/G-02_부품교체이력.csv",
    encoding="utf-8-sig"
)

print("부품교체이력 행 수:", len(chg))
print("교체이력 PART_CD 고유값 수:", chg["PART_CD"].nunique())

unmatched = chg.loc[
    ~chg["PART_CD"].isin(df["PART_CD"]),
    "PART_CD"
].nunique()

print("부품마스터에 없는 PART_CD 수:", unmatched)

In [ ]:
used_parts = chg["PART_CD"].nunique()
total_parts = df["PART_CD"].nunique()

print("전체 부품 수:", total_parts)
print("교체이력에 등장한 부품 수:", used_parts)
print("교체이력에 등장하지 않은 부품 수:", total_parts - used_parts)

## 최종 확인 결과

| 구분 | 확인 결과 |
|---|---|
| 데이터 크기 | 129행 × 6열 |
| 결측값 | 없음 |
| 중복 행 | 없음 |
| `PART_CD` | 129개 모두 고유하며 중복 없음 |
| `PART_TYPE` | 총 19개 유형 |
| 주요 `PART_TYPE` | 베어링류 14개, 열처리 12개, 동력전달 11개 |
| `APPLY_EQP_TYPE` | 총 26개 유형 |
| 주요 적용 설비 | 공통 57개, 연속로 11개, 컨베이어·압연기·용접기 각 5개 |
| `STD_LIFE_H` | 최소 500시간, 최대 89,000시간, 중앙값 15,500시간, 평균 약 20,953시간 |
| `LEAD_TIME_D` | 최소 5일, 최대 90일, 중앙값 29일, 평균 약 33.4일 |
| 수치형 변수 분포 | 표준수명과 리드타임 모두 일부 큰 값이 존재하는 오른쪽 꼬리 형태 |
| 부품교체이력 등장 부품 | 71개 |
| 부품교체이력 미등장 부품 | 58개 |
| 연결 키 | `PART_CD`를 이용해 부품교체이력과 연결 가능 |
| 시간 범위 | 부품마스터 자체에는 시간 컬럼이 없어 해당 없음 |
| 기존 TC/OP와 연결 | 시간 기준 직접 연결은 불가하며 부품교체이력을 통한 간접 연결 필요 |

### 결론

부품마스터는 결측·중복이 없고 `PART_CD`가 고유하게 관리되는 기준 데이터이다.  
부품별 표준수명과 조달 리드타임의 차이가 크므로 향후 정비·교체 분석에서 부품별 기준정보로 활용할 수 있다.